# Ambiente Big Data

**Disciplina**: Big Data  
**Trabalho**: Análise do SUSY Dataset com Apache Spark  
**Referência principal**: Baldi, P., Sadowski, P. & Whiteson, D. *Searching for exotic particles in high-energy physics with deep learning*. Nature Communications 5, 4308 (2014). DOI: 10.1038/ncomms5308

---

## Contexto do Projeto

Este trabalho aplica técnicas de Big Data ao dataset SUSY, um conjunto de dados gerado por simulações de colisões de partículas em colisionadores de alta energia (como o LHC/CERN). O objetivo é classificar eventos como sinal (produção de partículas superssimétricas) ou fundo (processos do Modelo Padrão sem nova física).

O paper de referência (Baldi et al., 2014) demonstrou que redes neurais profundas conseguem aprender automaticamente as mesmas representações que físicos precisaram construir manualmente, melhorando a classificação em até 8% sobre os métodos tradicionais. Nosso trabalho replica esse pipeline de análise usando Apache Spark para processar o dataset em escala.

---

## 1. Ambiente: Docker + Apache Spark

### Por que Docker?

O dataset SUSY tem aproximadamente 1.7 GB e mais de 5 milhões de linhas, um volume que demanda ferramentas de processamento distribuído. O Apache Spark resolve isso com processamento paralelo em memória. Para garantir reprodutibilidade do ambiente (versão do Spark, Java, Python e dependências), usamos a imagem `jupyter/pyspark-notebook`.

### Configuração do Container

**Opção 1: Comando direto:**

In [1]:
# Comando para subir o ambiente (executar no terminal do host, NÃO dentro do notebook)
# docker run --name spark-container -p 8889:8888 -p 4040:4040 -p 7077:7077 \
#   -v $(pwd):/home/jovyan/work jupyter/pyspark-notebook

# Mapeamento de portas:
#   8889 (host) → 8888 (container): Jupyter Notebook
#   4040 → Spark Web UI (monitoramento de jobs)
#   7077 → Spark Master (comunicação cluster)
#
# Porta 8888 do host estava ocupada por outro container (mlops-jupyter),
# por isso usamos 8889 como porta externa.
#
# -v $(pwd):/home/jovyan/work → monta o diretório atual dentro do container,
#   tornando o dataset e os notebooks acessíveis sem copiar arquivos.

print("Container iniciado via docker compose up")

Container iniciado via docker compose up


**Opção 2: Docker Compose** (`docker-compose.yml` na raiz do projeto):

```bash
docker compose up
```

> **Nota**: versões recentes do Docker (>= 20.10) integram o Compose como plugin nativo; o comando é `docker compose` (com espaço), não `docker-compose` (com hífen). O binário standalone `docker-compose` foi descontinuado.

O `docker-compose.yml` encapsula os mesmos parâmetros do `docker run`, facilitando o reuso e documentação do ambiente no controle de versão.

---

## 2. Inicialização do Apache Spark

### Por que SparkSession?

A `SparkSession` é o ponto de entrada unificado do Spark desde a versão 2.0. Ela inicializa o contexto de execução distribuída e disponibiliza as APIs de DataFrame, Spark SQL e MLlib, todos os recursos que utilizaremos nas Partes 2 e 3.

No contexto do paper de Baldi et al. (2014), o processamento de 5 milhões de eventos simulados exige justamente essa capacidade de distribuição: cada "evento" é uma colisão simulada no colisionador, com 18 variáveis cinemáticas medidas ou derivadas.

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SUSY-AC2") \
    .getOrCreate()

print(f"Apache Spark versão: {spark.version}")
print(f"Python versão:       {spark.sparkContext.pythonVer}")
print(f"App name:            {spark.sparkContext.appName}")
print(f"Master:              {spark.sparkContext.master}")

Apache Spark versão: 3.5.0
Python versão:       3.11
App name:            SUSY-AC2
Master:              local[*]


### Spark Web UI

Após executar a célula acima, o Spark disponibiliza uma interface web em **http://localhost:4040** (mapeada pelo `-p 4040:4040` do Docker). Ela exibe:

- **Jobs**: execuções disparadas (cada `count()`, `show()`, `write()` é um job)
- **Stages**: divisão interna de cada job em etapas paralelas
- **Storage**: DataFrames em cache na memória
- **Executors**: recursos alocados para o processamento

Essa visibilidade é central em Big Data: ao processar 5M de linhas do SUSY dataset, o Spark UI permite identificar gargalos, tempo de cada estágio e uso de memória, informações que justificam escolhas de arquitetura como a conversão para Parquet feita na Parte 2.


In [3]:
# Verificação da configuração do Spark
print("=== Configurações relevantes do SparkContext ===")
configs = [
    "spark.executor.memory",
    "spark.driver.memory",
    "spark.sql.shuffle.partitions",
]
for key in configs:
    try:
        val = spark.conf.get(key)
    except Exception:
        val = "(padrão do sistema)"
    print(f"  {key}: {val}")

=== Configurações relevantes do SparkContext ===
  spark.executor.memory: (padrão do sistema)
  spark.driver.memory: (padrão do sistema)
  spark.sql.shuffle.partitions: 200


---

## 3. Verificação de Acesso ao Dataset

O dataset SUSY foi obtido do UCI Machine Learning Repository (originalmente publicado junto ao paper de Baldi et al., 2014). Contém 5.000.000 eventos simulados de colisões de partículas, cada um descrito por 19 colunas:

| Coluna | Nome | Tipo | Origem |
|--------|------|------|--------|
| 0 | `label` | int (0/1) | **Target**: 1 = sinal SUSY, 0 = fundo |
| 1–8 | features cinemáticas brutas | float | **Low-level**: medições diretas do detector (pT, η, φ, energia perdida) |
| 9–18 | variáveis derivadas | float | **High-level**: construídas por físicos para capturar invariantes do processo |

A distinção entre features low-level e high-level é central no paper: Baldi et al. mostram que redes neurais treinadas apenas com features low-level conseguem AUC comparável à de modelos que usam as features high-level construídas manualmente (Tabela 1 do paper). Exploraremos esse contraste na Parte 3.

In [4]:
import os

dataset_path = "./data/supersymmetry_dataset.csv"

if os.path.exists(dataset_path):
    size_gb = os.path.getsize(dataset_path) / (1024 ** 3)
    print(f"Dataset encontrado: {dataset_path}")
    print(f"Tamanho: {size_gb:.2f} GB")
else:
    print(f"AVISO: dataset nao encontrado em {dataset_path}")
    print("Verifique se o volume Docker esta montado corretamente (-v flag).")

Dataset encontrado: ./data/supersymmetry_dataset.csv
Tamanho: 1.61 GB


---

## 4. Leitura Inicial do Dataset

O CSV do SUSY dataset **possui linha de header**, confirmado pela execução: com `header=False` o `df.count()` retornou **5.000.001** (uma linha a mais) e o `show(5)` exibiu `SUSY | lepton 1 pT | ...` como primeira linha de dados, com schema todo `string` (o `inferSchema=True` viu o texto do header e inferiu string para todas as colunas).

Por isso, na Parte 2 usamos `header=True`, que pula o header e infere os tipos corretos (`double`). Os nomes das colunas são então padronizados via `toDF(*column_names)` seguindo a descrição do paper (Supplementary Table 1 de Baldi et al., 2014).

> **Nota**: o código abaixo usa `header=False` como registro do comportamento observado nesta Parte 1. A leitura definitiva com `header=True` está no notebook da Parte 2.

In [5]:
# Nomes das colunas conforme Baldi et al. (2014), Supplementary Table 1
# Colunas 1-8: features low-level (medições diretas do detector)
# Colunas 9-18: features high-level (derivadas por físicos de partículas)
column_names = [
    "label",
    # Low-level: léptons e energia transversa perdida
    "lepton1_pT", "lepton1_eta", "lepton1_phi",
    "lepton2_pT", "lepton2_eta", "lepton2_phi",
    "missing_energy_magnitude", "missing_energy_phi",
    # High-level: variáveis cinemáticas derivadas
    "MET_rel", "axial_MET", "M_R", "M_TR_2", "R",
    "MT2", "S_R", "M_Delta_R", "dPhi_r_b", "cos_theta_r1"
]

df = spark.read.csv(
    "./data/supersymmetry_dataset.csv",
    header=False,
    inferSchema=True
)
df = df.toDF(*column_names)

print(f"Linhas carregadas: {df.count():,}")
print(f"Colunas:           {len(df.columns)}")
df.printSchema()

Linhas carregadas: 5,000,001
Colunas:           19
root
 |-- label: string (nullable = true)
 |-- lepton1_pT: string (nullable = true)
 |-- lepton1_eta: string (nullable = true)
 |-- lepton1_phi: string (nullable = true)
 |-- lepton2_pT: string (nullable = true)
 |-- lepton2_eta: string (nullable = true)
 |-- lepton2_phi: string (nullable = true)
 |-- missing_energy_magnitude: string (nullable = true)
 |-- missing_energy_phi: string (nullable = true)
 |-- MET_rel: string (nullable = true)
 |-- axial_MET: string (nullable = true)
 |-- M_R: string (nullable = true)
 |-- M_TR_2: string (nullable = true)
 |-- R: string (nullable = true)
 |-- MT2: string (nullable = true)
 |-- S_R: string (nullable = true)
 |-- M_Delta_R: string (nullable = true)
 |-- dPhi_r_b: string (nullable = true)
 |-- cos_theta_r1: string (nullable = true)



In [6]:
# Primeiras linhas para inspeção visual
df.show(5, truncate=False)

+-----+------------------+-------------------+-------------------+------------------+--------------------+-------------------+------------------------+-------------------+------------------+---------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------+
|label|lepton1_pT        |lepton1_eta        |lepton1_phi        |lepton2_pT        |lepton2_eta         |lepton2_phi        |missing_energy_magnitude|missing_energy_phi |MET_rel           |axial_MET            |M_R               |M_TR_2            |R                 |MT2               |S_R               |M_Delta_R         |dPhi_r_b          |cos_theta_r1        |
+-----+------------------+-------------------+-------------------+------------------+--------------------+-------------------+------------------------+-------------------+------------------+---------------------+------------------+------------------+----------------

---

## Resumo da Parte 1

| Item | Status |
|------|--------|
| Container Docker com Spark | Documentado (`docker run` + `docker-compose.yml`) |
| `SparkSession` inicializada | Versão verificada |
| Spark Web UI (porta 4040) | Disponível após inicialização |
| Dataset acessível no container | Verificado via volume Docker |
| Leitura inicial do CSV | Schema e primeiras linhas validados |

**Próximo passo**: `notebook_parte2_eda.ipynb` com EDA completa, pré-processamento e conversão para Parquet.